
# STEP4. 타이타닉 데이터 RAG 챗봇

## 프로젝트 소개

Kaggle Titanic train.csv 데이터를 분석하고,
분석 결과를 근거로 답변하는 RAG 챗봇을 구현한다.

## 데이터 및 분석 근거
- 원본 데이터: data/train.csv
- 분석 대상: 891명
- 분석 사실표: rag/titanic_facts.md
- 근거 ID: F01~F12

## RAG 처리 과정

1. 타이타닉 분석 사실표 불러오기
2. 사실표를 근거 ID 단위로 분리
3. OpenAI 임베딩 생성
4. 벡터 저장소에 등록
5. 질문과 관련된 근거 검색
6. 검색한 근거를 바탕으로 AI 답변 생성
7. 답변 수치 및 근거 ID 검증

## 사용 기술
- Python / Pandas
- LangChain
- OpenAI API
- InMemoryVectorStore




## ① 책장 1 : 타이타닉 분석 사실표

타이타닉 학습용 데이터 891명을 분석하여
12개의 사실(F01~F12)을 정리했습니다.

이 사실표는 RAG 챗봇이 답변할 때 사용하는 근거 자료입니다.

| 근거 ID | 분석 항목 | 주요 내용 |
|---|---|---|
| F01 | 전체 분석 대상 | 891명 |
| F02 | 생존자 수 | 342명 |
| F05 | 여성 생존율 | 74.2% |
| F06 | 남성 생존율 | 18.9% |
| F07 | 1등급 생존율 | 63.0% |

### 데이터 출처

- 원본: Kaggle Titanic train.csv
- 분석 사실표: rag/titanic_facts.md
- 분석 범위: Kaggle 학습용 표본 891명

※ 타이타닉 전체 탑승객에 대한 통계가 아닙니다.


In [3]:

# 셀 3. 타이타닉 사실표 읽기 → Document 만들기

from pathlib import Path
import re
from langchain_core.documents import Document

# 사실표 파일 경로
ROOT = Path(r"C:\workspace_2\Olist")
fact_path = ROOT / "rag" / "titanic_facts.md"

# Markdown 사실표 불러오기
text = fact_path.read_text(encoding="utf-8")

# F01~F12를 각각 하나의 문서로 분리
blocks = re.split(r"(?=^## F\d{2}\s)", text, flags=re.MULTILINE)

fact_docs = []

for block in blocks:
    match = re.match(r"## (F\d{2})\s", block)

    if match:
        fact_docs.append(
            Document(
                page_content=block.strip(),
                metadata={"id": match.group(1)}
            )
        )

print(f"사실 조각: {len(fact_docs)}개")
print(fact_docs[0].metadata)
print(fact_docs[0].page_content)


사실 조각: 12개
{'id': 'F01'}
## F01 전체 분석 대상
- 내용: 분석 대상은 891명이다.
- 기준: Kaggle Titanic train.csv 학습용 데이터
- 출처: data/train.csv



## ② 임베딩 및 벡터 저장소 만들기

타이타닉 분석 사실표의 12개 근거(F01~F12)를
임베딩하여 벡터 저장소에 등록합니다.

### 사용 기술
- OpenAIEmbeddings: 문서를 숫자 벡터로 변환
- InMemoryVectorStore: 변환된 벡터 저장
- similarity_search: 질문과 관련된 근거 검색

### 검색 과정
1. 사실표 12개를 임베딩
2. 벡터 저장소 생성
3. 질문을 입력
4. 관련 근거 3개 검색
5. 근거 ID와 내용 확인


In [4]:

# 셀 4. 타이타닉 벡터 저장소 생성

import os
from getpass import getpass

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

# 1. API 키 확인
api_key = os.getenv("OPENAI_API_KEY")


# 1. .env 파일에서 API 키 읽기
from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(".env 파일에서 OpenAI API 키를 찾지 못했습니다.")


# 2. 임베딩 모델 설정
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

# 3. 사실표 12개를 벡터 저장소에 등록
store = InMemoryVectorStore.from_documents(
    fact_docs,
    embeddings
)

print("벡터 저장소 생성 완료")
print(f"등록된 근거: {len(fact_docs)}개")


벡터 저장소 생성 완료
등록된 근거: 12개


In [5]:
# 셀 5. 질문과 관련된 근거 검색

question = "여성 승객의 생존율은 얼마야?"

results = store.similarity_search(question, k=3)

print(f"질문: {question}")

for doc in results:
    print("\n--------------------")
    print(f"근거 ID: {doc.metadata['id']}")
    print(doc.page_content)

질문: 여성 승객의 생존율은 얼마야?

--------------------
근거 ID: F06
## F06 남성 생존율
- 내용: 남성 577명 중 109명이 생존했으며 생존율은 18.9%이다.
- 기준: Kaggle Titanic train.csv 학습용 데이터
- 출처: data/train.csv

--------------------
근거 ID: F05
## F05 여성 생존율
- 내용: 여성 314명 중 233명이 생존했으며 생존율은 74.2%이다.
- 기준: Kaggle Titanic train.csv 학습용 데이터
- 출처: data/train.csv

--------------------
근거 ID: F09
## F09 3등급 생존율
- 내용: 3등급 491명 중 119명이 생존했으며 생존율은 24.2%이다.
- 기준: Kaggle Titanic train.csv 학습용 데이터
- 출처: data/train.csv


In [ ]:
# 셀 6. 찾기만 먼저 보기 : 같은 질문, 채널마다 어떤 조각을 찾아오나?
def search(question, channel, k=4):
    for doc, score in stores[channel].similarity_search_with_score(question, k=k):
        print(f"  [{doc.metadata['id']}] 유사도 {score:.2f} | {doc.page_content[:50]}…")

q = "이탈 모델은 어떤 숫자를 보고 판단해?"
for ch in ["A", "B", "C"]:
    print(f"채널 {ch} :")
    search(q, ch)


## ④ 지침(프롬프트) + 답하기 + 근거 ID 점검

In [6]:

# 셀 6. AI 답변 생성 + 근거 ID 검증

import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

NOT_FOUND = "제공된 자료로는 확인할 수 없습니다."

# 1. 답변 지침 설정
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 타이타닉 데이터 분석 Q&A 챗봇이야.\n"
     "자료는 Kaggle Titanic train.csv 학습용 데이터야.\n"
     "- 아래 [근거]에 있는 내용으로만 답해.\n"
     "- 질문 대상과 정확히 일치하는 근거만 사용해.\n"
     "- 비슷하지만 다른 항목의 숫자를 답하지 마.\n"
     "- 답변에 사용한 근거 ID를 [F05]처럼 붙여.\n"
     "- 숫자는 근거의 숫자를 그대로 사용해.\n"
     f"- 근거로 답할 수 없다면 '{NOT_FOUND}'라고만 답해.\n"
     "- 한국어로 3문장 이내로 답해."),
    ("human", "[근거]\n{context}\n\n[질문]\n{question}")
])

# 2. AI 모델 설정
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=api_key,
    temperature=0
)

chain = prompt | llm | StrOutputParser()

# 3. 검색 → 답변 → 근거 ID 검증
def ask(question, k=3):
    found = store.similarity_search(question, k=k)

    context = "\n".join(
        f"[{d.metadata['id']}] {d.page_content}"
        for d in found
    )

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    found_ids = {d.metadata["id"] for d in found}
    cited_ids = set(re.findall(r"\[(F\d{2})\]", answer))
    fake_ids = sorted(cited_ids - found_ids)

    return answer, sorted(found_ids), fake_ids

# 4. 첫 번째 질문 테스트
question = "여성 승객의 생존율은 얼마야?"

answer, found, fake = ask(question)

print("질문:", question)
print("AI 답변:", answer)
print("검색된 근거:", found)
print("찾지 않은 근거 ID 인용:", fake or "없음")


질문: 여성 승객의 생존율은 얼마야?
AI 답변: 여성 승객의 생존율은 74.2%입니다[F05].
검색된 근거: ['F05', 'F06', 'F09']
찾지 않은 근거 ID 인용: 없음


`fake`가 비어 있어야 정상입니다. 지침(부탁)만 믿지 않고 **코드로 한 번 더 확인**하는 안전장치입니다.

## ⑤ 타이타닉 질문별 답변 검증

In [7]:

# 셀 7. 타이타닉 질문별 AI 답변 검증

import pandas as pd

questions = [
    "여성 승객의 생존율은 얼마야?",
    "남성 승객의 생존율은 얼마야?",
    "1등급 승객의 생존율은 얼마야?",
    "3등급 승객의 생존율은 얼마야?",
    "승객의 국적별 생존율은 얼마야?"
]

rows = []

for q in questions:
    answer, found, fake = ask(q, k=3)

    rows.append({
        "질문": q,
        "AI 답변": answer,
        "검색된 근거": ", ".join(found),
        "가짜 근거 ID": ", ".join(fake) if fake else "없음"
    })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows))


,질문,AI 답변,검색된 근거,가짜 근거 ID
0,여성 승객의 생존율은 얼마야?,여성 승객의 생존율은 74.2%입니다. [F05],"F05, F06, F09",없음
1,남성 승객의 생존율은 얼마야?,남성 승객의 생존율은 18.9%입니다. [F06],"F05, F06, F09",없음
2,1등급 승객의 생존율은 얼마야?,1등급 승객의 생존율은 63.0%이다[F07].,"F07, F08, F09",없음
3,3등급 승객의 생존율은 얼마야?,3등급 승객의 생존율은 24.2%입니다.[F09],"F07, F08, F09",없음
4,승객의 국적별 생존율은 얼마야?,제공된 자료로는 확인할 수 없습니다.,"F05, F06, F07",없음



### 결과 읽는 법

| 검증 항목 | 확인 내용 |
|---|---|
| 근거 검색 | 질문에 적합한 사실표 ID를 찾았는가? |
| 답변 정확성 | 여성 74.2%, 남성 18.9% 등 정확한 수치를 답했는가? |
| 근거 표시 | 답변에 [F05]처럼 근거 ID가 표시됐는가? |
| 허위 근거 | 검색하지 않은 근거 ID를 인용하지 않았는가? |
| 답변 제한 | 자료에 없는 국적별 생존율을 임의로 생성하지 않았는가? |

**분석 결과**

- 타이타닉 분석 사실표 12개를 벡터 저장소에 등록했다.
- 5개 질문에 대한 챗봇 응답을 검증했다.
- 여성·남성·1등급·3등급 생존율을 원본 CSV와 대조했다.
- 검산한 4개 항목의 수치가 모두 일치했다.

**주의사항**

본 분석은 Kaggle Titanic train.csv의 891명을 대상으로 하며,
타이타닉 전체 탑승자에 대한 분석 결과가 아니다.


In [8]:

# 셀 8. 타이타닉 사실표 숫자 재검산

import pandas as pd

# 1. 원본 데이터 읽기
df = pd.read_csv(ROOT / "data" / "train.csv")

# 2. 검증할 항목
checks = [
    ("F05", "여성", df["Sex"] == "female", 314, 233, 74.2),
    ("F06", "남성", df["Sex"] == "male", 577, 109, 18.9),
    ("F07", "1등급", df["Pclass"] == 1, 216, 136, 63.0),
    ("F09", "3등급", df["Pclass"] == 3, 491, 119, 24.2)
]

# 3. 원본 데이터와 사실표 비교
rows = []

for fact_id, label, condition, expected_n, expected_s, expected_rate in checks:
    group = df.loc[condition]

    count = len(group)
    survived = int(group["Survived"].sum())
    rate = round(survived / count * 100, 1)

    match = (
        count == expected_n
        and survived == expected_s
        and rate == expected_rate
    )

    rows.append({
        "근거 ID": fact_id,
        "구분": label,
        "전체 인원": count,
        "생존자": survived,
        "생존율": f"{rate:.1f}%",
        "검증 결과": "일치" if match else "불일치"
    })

print(f"전체 분석 대상: {len(df)}명")
print(f"전체 생존자: {int(df['Survived'].sum())}명")

display(pd.DataFrame(rows))


전체 분석 대상: 891명
전체 생존자: 342명


,근거 ID,구분,전체 인원,생존자,생존율,검증 결과
0,F05,여성,314,233,74.2%,일치
1,F06,남성,577,109,18.9%,일치
2,F07,1등급,216,136,63.0%,일치
3,F09,3등급,491,119,24.2%,일치


사실표의 숫자(96,203건 · 8.1% · 3.0% · 4.30점/2.57점)와 같으면, 챗봇이 쓰는 근거를 **사람이 다시 확인한 것**입니다.

## 직접 해보기

| 단계 | 미션 |
|---|---|
| 기초 | 내가 만든 질문 3개를 세 채널에 던지고, 답에 붙은 ID를 사실표에서 찾아 대조 |
| 기본 | `k`를 2와 8로 바꿔 보고, 찾아오는 조각과 답이 어떻게 달라지는지 비교 |
| 심화 | 사실표에 내가 찾은 사실 3줄(F36~F38)을 추가하고, 그 사실을 묻는 질문에 답하는지 확인 |
| 도전 | 공개된 기관 규정이나 이용약관 PDF 하나로 셀 4-2를 다시 실행해 나만의 규정 챗봇 만들기 |

> 👉 같은 챗봇을 **Streamlit 대시보드의 '분석 Q&A' 메뉴**에서도 쓸 수 있습니다. (`utils/rag.py`가 이 노트북의 셀 3~7과 같은 코드)